In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import roc_auc_score
from rdkit.ML.Scoring.Scoring import CalcBEDROC
import os

In [ ]:

# for time in [2019]:
#     for cal_i in [0,1,2]:
#         if cal_i == 0:
#             filepath = f"/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_auc0.8.xlsx"
#             root = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_cv_rank_save_auc_norm_correct_pred'
#             files = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_cv_rank_save_auc_norm_correct'
#         elif cal_i == 1:
#             filepath = f"/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_all_fused.xlsx"
#             root = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_all_fused_pred'
#             files = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_all_fused'
        # elif cal_i == 2:
        #     filepath = f"/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_occ.xlsx"
        #     root = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_occ_pred'
        #     files = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_occ'

In [27]:
time = 2019
filepath = f"/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_auc0.8.xlsx"
root = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_cv_rank_save_auc_norm_correct_pred'
files = f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_cv_rank_save_auc_norm_correct'
all_disease_df_row = pd.read_excel(filepath, sheet_name="disease") 

In [28]:

valid_disease = all_disease_df_row['disease'].unique()

all_results = dict()
for file in os.listdir(root):
    filepath = os.path.join(root,file)
    disease_id = file[:9]
    if disease_id in valid_disease:
        with open(filepath, 'rb') as file:
            disease_pred = pickle.load(file)
        all_results[disease_id] = disease_pred

from model_reindex_fusion_weights_uniport_cv_filter import eval_bagging

def bedroc(y_true, y_pred):
    scores = np.column_stack((y_true, y_pred))  # Stack labels and scores as columns
    scores = scores[scores[:, 1].argsort()[::-1]]
    return [CalcBEDROC(scores, col=0, alpha=160.9), 
                            CalcBEDROC(scores, col=0, alpha=32.2),
                            CalcBEDROC(scores, col=0, alpha=16.1),
                            CalcBEDROC(scores, col=0, alpha=5.3)]
def recall_at_k(y_true, y_pred, k):
    # Sort indices of predictions in descending order
    top_k_indices = np.argsort(y_pred)[-k:][::-1]
    
    # Get true labels of top-k predicted samples
    top_k_true = y_true[top_k_indices]
    
    # Count true positives in top-k
    TP = np.sum(top_k_true == 1)
    
    # Total number of actual positives
    total_positives = np.sum(y_true == 1)
    
    # Handle edge case: no positives in ground truth
    if total_positives == 0:
        return 0.0
    
    recall = TP / total_positives
    return recall

def rank_weights(n):
    return [2 * (n - i + 1) / (n * (n + 1)) for i in range(1, n + 1)]

def run_eval(all_results, disease, ratio, fuse_features, fused_name):
    result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])
    a_dict = all_results[disease]
    threshold = int(ratio*len(a_dict['true_label']))
    fold = 1
    jac_sm = 0
    sort_dict = dict()

    for key in list(a_dict.keys())[1:]:
        sorted_indices = np.argsort(a_dict[key])[::-1]
        sort_dict[key] = sorted_indices
    weights = rank_weights(threshold)

    fused_rank = []
    for sample_index in range(len(a_dict['true_label'])):
        fused_sample_rank = 0
        # print('sample: ', sample_index)
        for key in fuse_features:
            top_ranks = sort_dict[key][:threshold]
            # print(top_ranks)
            if sample_index in top_ranks:
                single_rank = np.where(top_ranks == sample_index)[0][0]
                single_weighted_rank = weights[single_rank]
                fused_sample_rank += single_weighted_rank
                # print(single_rank,single_weighted_rank,fused_rank)
            else:
                fused_sample_rank += 0
        fused_rank.append(fused_sample_rank)
    ranked_predict_index, results = eval_bagging(np.array(fused_rank), a_dict['true_label'])
    result_df.loc[len(result_df.index)] = ["random_negative",fold,fused_name+'-'+str(round(jac_sm, 3)), *results]
    return result_df

ratio_list = [0.05,0.1,0.2]
# ratio_list = [0.1]

for disease in valid_disease:
    csv_path = os.path.join(files,disease+'.csv')
    df_existing = pd.read_csv(csv_path)
    df_existing = df_existing[~df_existing['para'].str.startswith('later')]

    all_eval = []

    for ratio in ratio_list:
        if time == 2019:
            fuse_features_dict = {'later_feature_fused_'+str(ratio): ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm'],
            'later_final_kernel_fused_'+str(ratio):['linear_fused','geo_fused']}
        elif time == 2017:
            fuse_features_dict = {'later_feature_fused_'+str(ratio): ['uniport_ppi_2017', 'ppi_2017_dw_80', 'uniport_exp','uniport_seq','uniport_esm'],
            'later_final_kernel_fused_'+str(ratio):['linear_fused','geo_fused']}

        for fused_name in list(fuse_features_dict.keys()):
            fuse_features = fuse_features_dict[fused_name]
            single_df = run_eval(all_results, disease, ratio, fuse_features, fused_name)
            # single_df['disease'] = disease
            all_eval.append(single_df)
    df_combined = pd.concat([df_existing, pd.concat(all_eval, ignore_index=True)], ignore_index=True)
    df_combined = df_combined.drop_duplicates()
    df_combined.to_csv(csv_path, index=False)
# combined_df = pd.concat(all_eval, ignore_index=True)

In [29]:
df_combined[df_combined['para']=='later_feature_fused_0.05-0']

,method,fold,para,top_recall_25,top_recall_300,top_recall_10%,top_precision_10%,max_precision_10%,top_recall_30%,top_precision_30%,...,pm_15%,pm_20%,pm_25%,pm_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30
8,random_negative,1,later_feature_fused_0.05-0,0.0,0.5,0.5,0.000639,0.001278,0.5,0.000213,...,0.769348,0.714377,0.666738,0.625055,0.714137,0.5009,0.38081,0.473482,0.486569,0.506447


In [194]:
feature_names = []
for i in list(next(iter(all_results.values())).keys()):
    feature_names.append(i)

In [196]:
feature_names

['true_label',
 'uniport_ppi_2017',
 'ppi_2017_dw_80',
 'uniport_exp',
 'uniport_seq',
 'uniport_esm',
 'early_fusion',
 'linear_fused',
 'geo_fused']

KeyboardInterrupt: 

In [209]:
df_combined

,method,fold,para,top_recall_25,top_recall_300,top_recall_10%,top_precision_10%,max_precision_10%,top_recall_30%,top_precision_30%,...,pm_15%,pm_20%,pm_25%,pm_30%,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30
0,random_negative,1,uniport_ppi_2017-0.297,0.036697,0.091743,0.229358,0.016779,0.073154,0.422018,0.010286,...,0.648677,0.636769,0.613112,0.585248,0.599523,0.4012,0.096569,0.118495,0.162836,0.304270
1,random_negative,1,ppi_2017_dw_80-0.313,0.036697,0.100917,0.238532,0.017450,0.073154,0.467890,0.011404,...,0.663443,0.654386,0.629173,0.610323,0.614091,0.3868,0.104639,0.125080,0.169419,0.317691
2,random_negative,1,uniport_exp-0.008,0.036697,0.073394,0.192661,0.014094,0.073154,0.467890,0.011404,...,0.656215,0.648703,0.639172,0.610323,0.594302,0.4064,0.074761,0.095304,0.139650,0.291491
3,random_negative,1,uniport_seq-0.176,0.000000,0.055046,0.247706,0.018121,0.073154,0.541284,0.013193,...,0.663443,0.665229,0.648656,0.644785,0.622310,0.3786,0.015935,0.089014,0.153117,0.322474
4,random_negative,1,uniport_esm-0.136,0.009174,0.045872,0.220183,0.016107,0.073154,0.522936,0.012746,...,0.683445,0.654386,0.623966,0.636743,0.621815,0.3791,0.033886,0.090393,0.150149,0.317515
5,random_negative,1,early_fusion-0.136,0.009174,0.045872,0.220183,0.016107,0.073154,0.522936,0.012746,...,0.683445,0.654386,0.623966,0.636743,0.621815,0.3791,0.033886,0.090393,0.150149,0.317515
6,random_negative,1,linear_fused-0.32,0.036697,0.100917,0.247706,0.018121,0.073154,0.504587,0.012299,...,0.695535,0.670404,0.639172,0.628336,0.624928,0.3760,0.104569,0.130839,0.179927,0.334188
7,random_negative,1,geo_fused-0.306,0.045872,0.100917,0.201835,0.014765,0.073154,0.559633,0.013640,...,0.683445,0.670404,0.662001,0.652486,0.653507,0.3476,0.102032,0.125259,0.172770,0.340241
8,random_negative,1,later_feature_fused_0.1-0,0.027523,0.110092,0.229358,0.016779,0.073154,0.513761,0.012522,...,0.689605,0.659892,0.653217,0.632587,0.627033,0.4415,0.085645,0.133766,0.183489,0.343082
9,random_negative,1,later_final_kernel_fused_0.1-0,0.036697,0.100917,0.201835,0.014765,0.073154,0.256881,0.006261,...,0.632587,0.562815,0.506838,0.461044,0.572899,0.5563,0.105684,0.127105,0.160482,0.287178


In [199]:
mean_df = combined_df.groupby('para').mean(numeric_only=True).reset_index()
for setting in combined_df['para'].unique():
    print(setting)
    filtred = combined_df[combined_df['para']==setting]
    ban_dict = dict()
    for select_metric in ['auroc','bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']:
        ban_list = []
        for disease in valid_disease:
            if filtred[filtred['disease'] == disease][select_metric].values[0] < all_disease_df_row[all_disease_df_row['disease'] == disease][select_metric].max():
                ban_list.append(disease)
        ban_dict[select_metric] = ban_list
    temp =[]
    for key in ban_dict:
        temp.append(len(ban_dict[key]))
    print(temp)

later_feature_fused_0.05-0
[50, 48, 49, 51, 52]
later_final_kernel_fused_0.05-0
[52, 43, 51, 53, 52]
later_feature_fused_0.1-0
[47, 48, 51, 48, 46]
later_final_kernel_fused_0.1-0
[49, 44, 50, 52, 53]
later_feature_fused_0.15-0
[46, 45, 49, 48, 46]
later_final_kernel_fused_0.15-0
[50, 44, 50, 51, 52]
later_feature_fused_0.2-0
[47, 44, 47, 46, 46]
later_final_kernel_fused_0.2-0
[49, 43, 49, 49, 48]


In [ ]:
2019_auc0.8
later_feature_fused_0.05-0
[42, 30, 41, 42, 42]
later_final_kernel_fused_0.05-0
[41, 35, 41, 42, 41]
later_feature_fused_0.1-0
[41, 31, 40, 42, 41]
later_final_kernel_fused_0.1-0
[40, 35, 40, 41, 41]
later_feature_fused_0.15-0
[41, 30, 40, 41, 40]
later_final_kernel_fused_0.15-0
[40, 35, 40, 41, 39]
later_feature_fused_0.2-0
[40, 30, 40, 40, 40]
later_final_kernel_fused_0.2-0
[40, 35, 40, 41, 39]
2017 all fused
later_feature_fused_0.05-0
[50, 48, 49, 51, 52]
later_final_kernel_fused_0.05-0
[52, 43, 51, 53, 52]
later_feature_fused_0.1-0
[47, 48, 51, 48, 46]
later_final_kernel_fused_0.1-0
[49, 44, 50, 52, 53]
later_feature_fused_0.15-0
[46, 45, 49, 48, 46]
later_final_kernel_fused_0.15-0
[50, 44, 50, 51, 52]
later_feature_fused_0.2-0
[47, 44, 47, 46, 46]
later_final_kernel_fused_0.2-0
[49, 43, 49, 49, 48]
2019 all_fused
later_feature_fused_0.05-0
[47, 36, 47, 48, 47]
later_final_kernel_fused_0.05-0
[47, 39, 46, 47, 47]
later_feature_fused_0.1-0
[46, 36, 46, 46, 45]
later_final_kernel_fused_0.1-0
[47, 39, 45, 47, 47]
later_feature_fused_0.15-0
[45, 35, 45, 45, 44]
later_final_kernel_fused_0.15-0
[46, 39, 43, 47, 45]
later_feature_fused_0.2-0
[43, 35, 43, 45, 43]
later_final_kernel_fused_0.2-0
[45, 39, 43, 47, 45]
2019_occ
later_feature_fused_0.05-0
[46, 34, 45, 47, 46]
later_final_kernel_fused_0.05-0
[47, 34, 46, 46, 48]
later_feature_fused_0.1-0
[46, 34, 45, 47, 46]
later_final_kernel_fused_0.1-0
[47, 34, 45, 47, 48]
later_feature_fused_0.15-0
[48, 34, 44, 47, 46]
later_final_kernel_fused_0.15-0
[45, 34, 45, 47, 48]
later_feature_fused_0.2-0
[47, 32, 42, 44, 47]
later_final_kernel_fused_0.2-0
[46, 34, 45, 47, 47]
2017 occ
later_feature_fused_0.05-0
[51, 35, 47, 51, 52]
later_final_kernel_fused_0.05-0
[51, 35, 48, 50, 51]
later_feature_fused_0.1-0
[49, 35, 47, 50, 53]
later_final_kernel_fused_0.1-0
[50, 35, 47, 50, 51]
later_feature_fused_0.15-0
[50, 34, 48, 52, 52]
later_final_kernel_fused_0.15-0
[50, 35, 46, 49, 51]
later_feature_fused_0.2-0
[51, 33, 47, 52, 53]
later_final_kernel_fused_0.2-0
[50, 35, 46, 50, 52]

SyntaxError: invalid syntax (1024230270.py, line 1)

In [ ]:
# print(roc_auc_score(a_dict['true_label'], fused_rank))
# print(roc_auc_score(a_dict['true_label'], a_dict['ppi_2019_dw_40']))
# print(bedroc(a_dict['true_label'], fused_rank))
# print(bedroc(a_dict['true_label'], a_dict['ppi_2019_dw_40']))
# print(recall_at_k(a_dict['true_label'], fused_rank,300))
# print(recall_at_k(a_dict['true_label'], a_dict['ppi_2019_dw_40'],300))